In [ ]:
# In[1]:

import pandas as pd

# Load CSVs into DataFrames and parse timestamps (UTC)
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Helper to produce count and min/max ISO timestamps
def times_summary(df):
    if df.shape[0] == 0:
        return {"rows": 0, "min_ts": None, "max_ts": None}
    return {
        "rows": int(df.shape[0]),
        "min_ts": df['timestamp'].min().isoformat(),
        "max_ts": df['timestamp'].max().isoformat()
    }

# 1) For each file: total rows and min/max timestamp
metric_overview = pd.DataFrame([times_summary(metric_df)])
trace_overview = pd.DataFrame([times_summary(trace_df)])
log_overview = pd.DataFrame([times_summary(log_df)])
error_overview = pd.DataFrame([times_summary(error_df)])

# 2) metric.csv: distinct cmdb_id values and top 20 (kpi_name, count)
metric_cmdb = pd.Series(sorted(metric_df['cmdb_id'].unique()))[:20]
metric_top = (
    metric_df.groupby('kpi_name', dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 3) trace.csv: distinct cmdb_id values and top 20 (trace_name, count)
trace_cmdb = pd.Series(sorted(trace_df['cmdb_id'].unique()))[:20]
trace_top = (
    trace_df.groupby('trace_name', dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 4) log.csv: distinct cmdb_id values and top 20 (log_name, count)
log_cmdb = pd.Series(sorted(log_df['cmdb_id'].unique()))[:20]
log_top = (
    log_df.groupby('log_name', dropna=False)
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
    .head(20)
    .reset_index(drop=True)
)

# 5) error_logs.csv: distinct cmdb_id values, total rows, and up to first 5 rows with truncated message
error_cmdb = pd.Series(sorted(error_df['cmdb_id'].unique()))[:20]
error_rows_total = int(error_df.shape[0])
# Prepare sample: show timestamp ISO, cmdb_id, truncated message (first 200 chars)
if error_df.shape[0] > 0:
    error_sample = (
        error_df[['timestamp', 'cmdb_id', 'message']]
        .head(5)
        .assign(
            timestamp_iso=lambda d: d['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%S%z').str.replace('+0000', '+00:00'),
            message_trunc=lambda d: d['message'].astype(str).str.slice(0, 200)
        )
        .loc[:, ['timestamp_iso', 'cmdb_id', 'message_trunc']]
        .reset_index(drop=True)
    )
else:
    error_sample = pd.DataFrame(columns=['timestamp_iso', 'cmdb_id', 'message_trunc'])

# Compact outputs (stored in variables for later steps)
metric_overview, metric_cmdb, metric_top, trace_overview, trace_cmdb, trace_top, log_overview, log_cmdb, log_top, error_overview, error_cmdb, error_rows_total, error_sample

```
Out[1]:
```
# Summary of telemetry files (compact, plain English)
summary = (
    "Files and time range:\n"
    "- metric.csv: 8450 rows; min timestamp 2024-01-24T01:43:00+00:00; max timestamp 2024-01-24T02:07:00+00:00.\n"
    "- trace.csv: 9920 rows; min timestamp 2024-01-24T01:43:00+00:00; max timestamp 2024-01-24T02:07:00+00:00.\n"
    "- log.csv: 1888 rows; min timestamp 2024-01-24T01:43:00+00:00; max timestamp 2024-01-24T02:07:00+00:00.\n"
    "- error_logs.csv: 0 rows (no error-log entries found).\n\n"
    "metric.csv highlights:\n"
    "- Example services present include ts-admin-basic-info-service, ts-admin-order-service, ts-auth-service, ts-order-service, ts-train-service, ts-travel-service, and many others.\n"
    "- Top kpi_name counts (top 20 shown; highest counts first):\n"
    "  * cpu: 1700\n"
    "  * mem: 1700\n"
    "  * socket: 1700\n"
    "  * diskio: 1675\n"
    "  * workload: 550\n"
    "  * latency-50: 525\n"
    "  * latency-90: 525\n"
    "  * error: 75\n\n"
    "trace.csv highlights:\n"
    "- Distinct trace sources include 'root' plus many ts-* services (e.g., ts-order-service, ts-train-service, ts-travel-service, ts-food-service, ts-station-service, etc.).\n"
    "- Top trace_name counts (top items):\n"
    "  * trace.from_root.duration_mean: 273\n"
    "  * trace.from_root.duration_p95: 273\n"
    "  * trace.from_root.error_rate: 273\n"
    "  * trace.from_root.row_count: 273\n"
    "  * trace.to_ts-station-service.row_count: 124\n"
    "  * trace.to_ts-station-service.error_rate: 124\n"
    "  * trace.to_ts-station-service.duration_p95: 124\n"
    "  * trace.to_ts-station-service.duration_mean: 124\n"
    "  * Groups of trace metrics for ts-food-service and ts-travel-service (counts ~100 each).\n\n"
    "log.csv highlights:\n"
    "- Distinct cmdb_ids include many ts-* services (similar to metric/trace lists).\n"
    "- Top log_name counts:\n"
    "  * log.error_count: 944\n"
    "  * log.row_count: 944\n\n"
    "error_logs.csv:\n"
    "- No rows present; therefore no sample messages to show.\n\n"
    "Overall quick takeaway:\n"
    "- All telemetry (metric/trace/log) covers the same short UTC window (2024-01-24T01:43:00 to 2024-01-24T02:07:00).\n"
    "- Metric KPIs are dominated by cpu/mem/socket/diskio signals; traces show strong 'from_root' and specific interactions with ts-station, ts-food, and ts-travel services.\n"
    "- Logs have aggregated error_count and row_count signals but there are no raw error_log entries.\n"
    "This summary is compact; full DataFrames remain available in kernel variables for deeper inspection if needed."
)

summary

The original code execution output of IPython Kernel is also provided below for reference:

(   rows                     min_ts                     max_ts
0  8450  2024-01-24T01:43:00+00:00  2024-01-24T02:07:00+00:00, 0     ts-admin-basic-info-service
1          ts-admin-order-service
2          ts-admin-route-service
3         ts-admin-travel-service
4           ts-admin-user-service
5              ts-assurance-mongo
6            ts-assurance-service
7                   ts-auth-mongo
8                 ts-auth-service
9               ts-avatar-service
10               ts-basic-service
11              ts-cancel-service
12                ts-config-mongo
13              ts-config-service
14               ts-consign-mongo
15         ts-consign-price-mongo
16       ts-consign-price-service
17             ts-consign-service
18              ts-contacts-mongo
19            ts-contacts-service
dtype: object,      kpi_name  count
0         cpu   1700
1         mem   1700
2      socket   1700
3      diskio   1675
4    workload    550
5  latency-50    525
6  latency-90    525
7       error     75,    rows                     min_ts                     max_ts
0  9920  2024-01-24T01:43:00+00:00  2024-01-24T02:07:00+00:00, 0                            root
1     ts-admin-basic-info-service
2         ts-admin-travel-service
3            ts-assurance-service
4                 ts-auth-service
5                ts-basic-service
6               ts-config-service
7        ts-consign-price-service
8              ts-consign-service
9             ts-food-map-service
10                ts-food-service
11      ts-inside-payment-service
12         ts-order-other-service
13               ts-order-service
14               ts-price-service
15               ts-route-service
16             ts-station-service
17          ts-ticketinfo-service
18               ts-train-service
19              ts-travel-service
dtype: object,                                      trace_name  count
0                 trace.from_root.duration_mean    273
1                  trace.from_root.duration_p95    273
2                    trace.from_root.error_rate    273
3                     trace.from_root.row_count    273
4         trace.to_ts-station-service.row_count    124
5        trace.to_ts-station-service.error_rate    124
6      trace.to_ts-station-service.duration_p95    124
7     trace.to_ts-station-service.duration_mean    124
8      trace.from_ts-food-service.duration_mean    100
9       trace.from_ts-food-service.duration_p95    100
10        trace.from_ts-food-service.error_rate    100
11         trace.from_ts-food-service.row_count    100
12       trace.from_ts-travel-service.row_count    100
13      trace.from_ts-travel-service.error_rate    100
14    trace.from_ts-travel-service.duration_p95    100
15   trace.from_ts-travel-service.duration_mean    100
16      trace.from_ts-travel2-service.row_count    100
17     trace.from_ts-travel2-service.error_rate    100
18   trace.from_ts-travel2-service.duration_p95    100
19  trace.from_ts-travel2-service.duration_mean    100,    rows                     min_ts                     max_ts
0  1888  2024-01-24T01:43:00+00:00  2024-01-24T02:07:00+00:00, 0     ts-admin-basic-info-service
1         ts-admin-travel-service
2              ts-assurance-mongo
3            ts-assurance-service
4                   ts-auth-mongo
5                 ts-auth-service
6                ts-basic-service
7                 ts-config-mongo
8                ts-consign-mongo
9          ts-consign-price-mongo
10             ts-consign-service
11              ts-contacts-mongo
12              ts-food-map-mongo
13            ts-food-map-service
14                  ts-food-mongo
15                ts-food-service
16        ts-inside-payment-mongo
17      ts-inside-payment-service
18          ts-notification-mongo
19                 ts-order-mongo
dtype: object,           log_name  count
0  log.error_count    944
1    log.row_count    944,    rows min_ts max_ts
0     0   None   None, Series([], dtype: object), 0, Empty DataFrame
Columns: [timestamp_iso, cmdb_id, message_trunc]
Index: [])```
```

In [ ]:
# In[2]:

import numpy as np
import pandas as pd

# Reuse metric_df already loaded in kernel; filter to candidate components
candidates = [
    "ts-auth-service",
    "ts-order-service",
    "ts-route-service",
    "ts-train-service",
    "ts-travel-service",
]

metric_candidates_df = metric_df[metric_df['cmdb_id'].isin(candidates)].copy()

# Incident window (UTC)
start_window = pd.to_datetime("2024-01-24 01:40:52", utc=True)
end_window = pd.to_datetime("2024-01-24 02:10:52", utc=True)

# Group by cmdb_id and kpi_name using the full available time series
group = metric_candidates_df.groupby(['cmdb_id', 'kpi_name'])

agg_count = group.size().rename('count')
agg_min = group['value'].min().rename('min')
agg_max = group['value'].max().rename('max')
agg_mean = group['value'].mean().rename('mean')
agg_p95 = group['value'].quantile(0.95).rename('p95')  # global P95 on entire series (rule 9)

agg_df = pd.concat([agg_count, agg_min, agg_max, agg_mean, agg_p95], axis=1).reset_index()

# Compute how many points fall inside the incident window for each pair
window_df = metric_candidates_df[
    (metric_candidates_df['timestamp'] >= start_window) &
    (metric_candidates_df['timestamp'] <= end_window)
]

window_counts = window_df.groupby(['cmdb_id', 'kpi_name']).size().rename('count_in_window').reset_index()

# Merge window counts into agg_df
final_agg = (
    agg_df.merge(window_counts, on=['cmdb_id', 'kpi_name'], how='left')
    .fillna({'count_in_window': 0})
)

# Ensure integer type for counts
final_agg['count'] = final_agg['count'].astype(int)
final_agg['count_in_window'] = final_agg['count_in_window'].astype(int)

# Sort by global count descending and keep top 50 (compact)
final_agg = final_agg.sort_values('count', ascending=False).reset_index(drop=True).head(50)

# Round numeric summaries for compact display
final_agg['mean'] = final_agg['mean'].round(6)
final_agg['p95'] = final_agg['p95'].round(6)
final_agg['min'] = final_agg['min'].round(6)
final_agg['max'] = final_agg['max'].round(6)

# Output: final aggregated table (compact) and keep metric_candidates_df stored for later steps
final_agg

```
Out[2]:
```
Summary of the requested aggregates (plain English):

- I computed aggregates for the five candidate components: ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service. There are 35 (cmdb_id, kpi_name) pairs in the result.

- Data coverage:
  - Every (cmdb_id, kpi_name) pair has 25 samples (count = 25).
  - For every pair, all 25 points fall inside the incident window 2024-01-24 01:40:52 — 2024-01-24 02:10:52 UTC (count_in_window = 25). In other words, the available time series for these components fully overlaps the incident window.

- Example aggregates (representative samples from the table):
  - ts-auth-service / cpu: count 25, min 13.21481, max 17.59565, mean 14.82358, P95 17.15472.
  - ts-auth-service / diskio: count 25, min 40579.88, max 156069.9, mean 79740.87, P95 130625.3.
  - ts-auth-service / mem: count 25, mean ~2.485663e+08, P95 ~2.493700e+08.
  - ts-travel-service / latency-50: count 25, min 0.034408, max 0.05108, mean 0.041506, P95 0.050223.
  - ts-travel-service / mem: count 25, mean ~2.700133e+08, P95 ~2.705936e+08.

- Overall takeaway:
  - All requested KPIs for the five candidate services are present and fully cover the incident window (no missing samples inside the window).
  - ts-auth-service shows relatively high CPU and disk I/O values (see example stats above). ts-travel-service shows low latency values and large memory footprint in the examples.
  - The full detailed aggregated table is available in the kernel variable final_agg, and the filtered metric DataFrame for these five components is stored as metric_candidates_df for follow-up analysis.

If you want, I can (next) rank these pairs by P95 or produce per-service summaries highlighting the most abnormal KPIs. Which follow-up would you like?

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name  count           min           max          mean           p95  count_in_window
0     ts-auth-service         cpu     25  1.321481e+01  1.759565e+01  1.482358e+01  1.715472e+01               25
1     ts-auth-service      diskio     25  4.057988e+04  1.560699e+05  7.974087e+04  1.306253e+05               25
2     ts-auth-service  latency-50     25  2.056110e-01  3.071210e-01  2.332700e-01  2.655570e-01               25
3     ts-auth-service  latency-90     25  4.517750e-01  9.170060e-01  6.386850e-01  8.434950e-01               25
4     ts-auth-service         mem     25  2.470257e+08  2.494731e+08  2.485663e+08  2.493700e+08               25
..                ...         ...    ...           ...           ...           ...           ...              ...
30  ts-travel-service  latency-50     25  3.440800e-02  5.108000e-02  4.150600e-02  5.022300e-02               25
31  ts-travel-service  latency-90     25  8.450800e-02  9.388600e-02  8.854100e-02  9.235100e-02               25
32  ts-travel-service         mem     25  2.695240e+08  2.706343e+08  2.700133e+08  2.705936e+08               25
33  ts-travel-service      socket     25  1.133333e+01  1.415000e+01  1.263493e+01  1.351333e+01               25
34  ts-travel-service    workload     25  1.396567e+00  1.984433e+00  1.687668e+00  1.946987e+00               25

[35 rows x 8 columns]

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[3]:

import numpy as np
import pandas as pd

# Reuse existing variables: metric_candidates_df, final_agg
# Incident window (UTC)
start_window = pd.to_datetime("2024-01-24 01:40:52", utc=True)
end_window = pd.to_datetime("2024-01-24 02:10:52", utc=True)

# Ensure final_agg has p95 available for all (cmdb_id,kpi_name)
p95_map = final_agg.set_index(['cmdb_id', 'kpi_name'])['p95']

# Filter metric rows to incident window
window_df = metric_candidates_df[
    (metric_candidates_df['timestamp'] >= start_window) &
    (metric_candidates_df['timestamp'] <= end_window)
].copy()

# Attach global p95 to window rows
window_df = window_df.merge(
    final_agg[['cmdb_id', 'kpi_name', 'p95']],
    on=['cmdb_id', 'kpi_name'],
    how='left'
)

# Mark anomalies where value >= global p95 (use >= per instruction)
window_df['is_anomaly'] = window_df['value'] >= window_df['p95']

# Aggregate per (cmdb_id, kpi_name) inside the window
agg = window_df.groupby(['cmdb_id', 'kpi_name']).agg(
    count_in_window = ('value', 'size'),
    count_anomaly = ('is_anomaly', 'sum'),
    max_value_in_window = ('value', 'max'),
    earliest_anom_ts = ('timestamp', lambda s: s[window_df.loc[s.index, 'is_anomaly']].min() if s.size>0 else pd.NaT)
).reset_index()

# Bring p95 from final_agg (global) into the result
agg = agg.merge(final_agg[['cmdb_id', 'kpi_name', 'p95']], on=['cmdb_id', 'kpi_name'], how='left')

# Compute severity = max_value_in_window / p95 (float). Handle p95 == 0.
def compute_severity(row):
    p = row['p95']
    if pd.isna(p) or p == 0:
        return float('inf') if row['max_value_in_window'] > 0 else 0.0
    return float(row['max_value_in_window']) / float(p)

agg['severity'] = agg.apply(compute_severity, axis=1)

# Format earliest anomaly timestamp as ISO UTC string, or None
def to_iso_or_none(ts):
    if pd.isna(ts):
        return None
    # ts is timezone-aware UTC; format as ISO with +00:00
    return ts.strftime('%Y-%m-%dT%H:%M:%S%z').replace('+0000', '+00:00')

agg['earliest_anomaly_timestamp'] = agg['earliest_anom_ts'].apply(to_iso_or_none)

# Rename columns to match requested names and select/order columns
metric_anomalies_df = agg[[
    'cmdb_id',
    'kpi_name',
    'count_in_window',
    'count_anomaly',
    'earliest_anomaly_timestamp',
    'max_value_in_window',
    'p95',
    'severity'
]].copy()

# Ensure numeric rounding for compactness
metric_anomalies_df['max_value_in_window'] = metric_anomalies_df['max_value_in_window'].round(6)
metric_anomalies_df['p95'] = metric_anomalies_df['p95'].round(6)
metric_anomalies_df['severity'] = metric_anomalies_df['severity'].astype(float).round(6)

# Top 20 rows sorted by severity descending
metric_anomalies_top20 = metric_anomalies_df.sort_values('severity', ascending=False).head(20).reset_index(drop=True)

# Full list of (cmdb_id, kpi_name) with count_anomaly > 0 (names + count_anomaly), limit top 50 by count_anomaly desc
anomalous_pairs = (
    metric_anomalies_df[metric_anomalies_df['count_anomaly'] > 0]
    .loc[:, ['cmdb_id', 'kpi_name', 'count_anomaly']]
    .sort_values(['count_anomaly', 'cmdb_id', 'kpi_name'], ascending=[False, True, True])
    .head(50)
    .reset_index(drop=True)
)

# Display compact outputs (variables)
metric_anomalies_top20, anomalous_pairs

```
Out[3]:
```
Summary of metric anomaly detection inside the incident window (2024-01-24 01:40:52 — 2024-01-24 02:10:52 UTC):

High-level
- Number of (cmdb_id, kpi_name) pairs examined (for the 5 candidate services): 35.
- Number of pairs with at least one anomaly (value >= global P95): 35 (i.e., every examined pair had >=1 point >= its global P95).
- Most pairs have count_in_window = 25 (full coverage of the incident window) and most anomalies are few points (typically 2 anomalies per pair; one pair had 1).

Top anomalies by severity (severity = max_value_in_window / global P95). Top 10 shown (severity, earliest anomaly time, anomaly count):
1. ts-train-service / cpu — severity 1.868329; earliest anomaly 2024-01-24T01:47:00+00:00; count_anomaly=2
2. ts-auth-service / diskio — severity 1.19479; earliest anomaly 2024-01-24T01:58:00+00:00; count_anomaly=2
3. ts-order-service / workload — severity 1.194136; earliest anomaly 2024-01-24T01:52:00+00:00; count_anomaly=2
4. ts-auth-service / latency-50 — severity 1.156516; earliest anomaly 2024-01-24T01:50:00+00:00; count_anomaly=2
5. ts-travel-service / cpu — severity 1.15467; earliest anomaly 2024-01-24T01:43:00+00:00; count_anomaly=2
6. ts-order-service / latency-50 — severity 1.111122; earliest anomaly 2024-01-24T01:57:00+00:00; count_anomaly=2
7. ts-auth-service / latency-90 — severity 1.08715; earliest anomaly 2024-01-24T01:48:00+00:00; count_anomaly=2
8. ts-train-service / workload — severity 1.081898; earliest anomaly 2024-01-24T01:45:00+00:00; count_anomaly=2
9. ts-train-service / latency-90 — severity 1.06561; earliest anomaly 2024-01-24T01:47:00+00:00; count_anomaly=1
10. ts-travel-service / socket — severity 1.047114; earliest anomaly 2024-01-24T01:44:00+00:00; count_anomaly=2

Other notable items (observed among top entries):
- Disk I/O anomalies at ts-auth-service and ts-route-service (severity ≈1.19 and ≈1.04 respectively).
- Several socket and CPU KPIs across ts-auth, ts-train, ts-travel, and ts-order show mild-to-moderate severity (1.02–1.15).
- For almost all pairs, only a small number of samples exceed the global P95 (usually 2 points out of 25).

Stored results
- The detailed anomalies table is saved in kernel variable metric_anomalies_df.
- The filtered metric rows for the five candidate services remain in metric_candidates_df.

Suggested next steps
- Inspect the anomaly timestamps and raw values for the top-ranked pairs in metric_anomalies_df to confirm spike patterns and correlate with traces/logs.
- Check trace and log telemetry at the earliest anomaly timestamps for related errors or latency spikes.

If you want, I can: show the full anomalies table (metric_anomalies_df.head(N)), list all pairs with their count_anomaly, or extract the raw metric rows for any top pair for manual inspection. Which would you like next?

The original code execution output of IPython Kernel is also provided below for reference:

(              cmdb_id    kpi_name  count_in_window  count_anomaly earliest_anomaly_timestamp  max_value_in_window            p95  severity
0    ts-train-service         cpu               25              2  2024-01-24T01:47:00+00:00             1.606954       0.860102  1.868329
1     ts-auth-service      diskio               25              2  2024-01-24T01:58:00+00:00        156069.874993  130625.317932  1.194790
2    ts-order-service    workload               25              2  2024-01-24T01:52:00+00:00             0.632283       0.529490  1.194136
3     ts-auth-service  latency-50               25              2  2024-01-24T01:50:00+00:00             0.307121       0.265557  1.156516
4   ts-travel-service         cpu               25              2  2024-01-24T01:43:00+00:00             1.422031       1.231547  1.154670
5    ts-order-service  latency-50               25              2  2024-01-24T01:57:00+00:00             0.041788       0.037609  1.111122
6     ts-auth-service  latency-90               25              2  2024-01-24T01:48:00+00:00             0.917006       0.843495  1.087150
7    ts-train-service    workload               25              2  2024-01-24T01:45:00+00:00             3.799950       3.512300  1.081898
8    ts-train-service  latency-90               25              1  2024-01-24T01:47:00+00:00             0.009502       0.008917  1.065610
9   ts-travel-service      socket               25              2  2024-01-24T01:44:00+00:00            14.150000      13.513333  1.047114
10   ts-route-service      diskio               25              2  2024-01-24T01:44:00+00:00         65498.026265   62708.799723  1.044479
11   ts-train-service      socket               25              2  2024-01-24T01:55:00+00:00             9.850000       9.453333  1.041961
12   ts-order-service      socket               25              2  2024-01-24T01:57:00+00:00            13.200000      12.746667  1.035565
13  ts-travel-service      diskio               25              2  2024-01-24T01:51:00+00:00         61817.225848   59879.465834  1.032361
14    ts-auth-service      socket               25              2  2024-01-24T01:48:00+00:00            17.450000      16.933333  1.030512
15   ts-route-service      socket               25              2  2024-01-24T01:50:00+00:00            15.733333      15.320000  1.026980
16   ts-train-service      diskio               25              2  2024-01-24T01:53:00+00:00         61582.882322   60014.688969  1.026130
17    ts-auth-service         cpu               25              2  2024-01-24T01:51:00+00:00            17.595645      17.154716  1.025703
18   ts-train-service  latency-50               25              2  2024-01-24T01:43:00+00:00             0.004448       0.004363  1.019475
19  ts-travel-service    workload               25              2  2024-01-24T01:53:00+00:00             1.984433       1.946987  1.019233,               cmdb_id    kpi_name  count_anomaly
0     ts-auth-service         cpu              2
1     ts-auth-service      diskio              2
2     ts-auth-service  latency-50              2
3     ts-auth-service  latency-90              2
4     ts-auth-service         mem              2
..                ...         ...            ...
30  ts-travel-service  latency-90              2
31  ts-travel-service         mem              2
32  ts-travel-service      socket              2
33  ts-travel-service    workload              2
34   ts-train-service  latency-90              1

[35 rows x 3 columns])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[4]:

import pandas as pd

# Reuse previously defined variables where possible
# candidates list and incident window
candidates = [
    "ts-auth-service",
    "ts-order-service",
    "ts-route-service",
    "ts-train-service",
    "ts-travel-service",
]

start_window = pd.to_datetime("2024-01-24 01:40:52", utc=True)
end_window = pd.to_datetime("2024-01-24 02:10:52", utc=True)

# Filter trace_df to the incident window
trace_window = trace_df[
    (trace_df['timestamp'] >= start_window) &
    (trace_df['timestamp'] <= end_window)
].copy()

# Select rows where trace_name mentions any of the patterns and references one of the candidate services
pattern_main = r'(from_root|from_ts-|to_ts-)'
pattern_services = '|'.join([pd.util.escape_string(s) for s in candidates]) if hasattr(pd.util, 'escape_string') else '|'.join(candidates)

mask_main = trace_window['trace_name'].str.contains(pattern_main, na=False)
mask_service = trace_window['trace_name'].str.contains(pattern_services, na=False)

trace_candidates_df = trace_window[mask_main & mask_service].copy()

# Aggregate per distinct trace_name: count, max_value, earliest_timestamp_of_max (ISO UTC), mean_value
def agg_trace(group):
    cnt = len(group)
    maxv = group['value'].max()
    # earliest timestamp where value == max
    ts_max = group.loc[group['value'] == maxv, 'timestamp'].min()
    meanv = group['value'].mean()
    return pd.Series({
        'count': int(cnt),
        'max_value': float(maxv),
        'earliest_timestamp_of_max': ts_max,
        'mean_value': float(meanv)
    })

trace_agg_df = trace_candidates_df.groupby('trace_name', dropna=False).apply(agg_trace).reset_index()

# Format earliest timestamp as ISO UTC string (or None)
trace_agg_df['earliest_timestamp_of_max'] = trace_agg_df['earliest_timestamp_of_max'].apply(
    lambda t: None if pd.isna(t) else t.strftime('%Y-%m-%dT%H:%M:%S%z').replace('+0000', '+00:00')
)

# Round numeric columns for compactness
trace_agg_df['max_value'] = trace_agg_df['max_value'].round(6)
trace_agg_df['mean_value'] = trace_agg_df['mean_value'].round(6)

# Sort by max_value descending and limit to top 20
trace_agg_top20 = trace_agg_df.sort_values('max_value', ascending=False).head(20).reset_index(drop=True)

# Find trace_names among these whose metric is an error_rate and has max_value > 0
trace_error_rates = trace_agg_df[
    trace_agg_df['trace_name'].str.contains('error_rate', na=False) &
    (trace_agg_df['max_value'] > 0)
].loc[:, ['trace_name', 'max_value']].sort_values('max_value', ascending=False).reset_index(drop=True)

# Return compact outputs (keep full aggregates and filtered traces stored in variables)
trace_agg_top20, trace_error_rates, trace_candidates_df.shape, trace_agg_df.shape

```
Out[4]:
```
Summary (incident window 2024-01-24 01:40:52 — 2024-01-24 02:10:52 UTC):

- I filtered trace rows inside the window and selected trace_names matching from_root / from_ts- / to_ts- that reference the candidate services. The filtered trace DataFrame is stored as trace_candidates_df (shape: 2380 rows, 4 columns). The aggregate results are in trace_agg_df (40 distinct trace_name rows).

- Top trace_name aggregates by max_value (top 20, showing count, max_value, earliest_timestamp_of_max, mean_value):
  1. trace.to_ts-route-service.row_count — count 75, max 746, earliest_max 2024-01-24T01:51:00+00:00, mean 333.44
  2. trace.from_ts-route-service.row_count — count 25, max 746, earliest_max 2024-01-24T01:51:00+00:00, mean 666.88
  3. trace.from_ts-train-service.row_count — count 25, max 440, earliest_max 2024-01-24T01:44:00+00:00, mean 369.60
  4. trace.to_ts-train-service.row_count — count 75, max 440, earliest_max 2024-01-24T01:44:00+00:00, mean 154.00
  5. trace.from_ts-auth-service.row_count — count 25, max 392, earliest_max 2024-01-24T01:51:00+00:00, mean 341.64
  6. trace.from_ts-travel-service.row_count — count 100, max 392, earliest_max 2024-01-24T01:53:00+00:00, mean 135.18
  7. trace.to_ts-travel-service.row_count — count 100, max 392, earliest_max 2024-01-24T01:53:00+00:00, mean 91.99
  8. trace.to_ts-auth-service.row_count — count 50, max 392, earliest_max 2024-01-24T01:51:00+00:00, mean 213.58
  9. trace.from_ts-order-service.row_count — count 48, max 63, earliest_max 2024-01-24T01:52:00+00:00, mean 20.416667
 10. trace.to_ts-order-service.row_count — count 72, max 63, earliest_max 2024-01-24T01:52:00+00:00, mean 15.027778
 11. trace.to_ts-auth-service.duration_p95 — count 50, max 1.11233, earliest_max 2024-01-24T01:47:00+00:00, mean 0.331168
 12. trace.to_ts-auth-service.duration_mean — count 50, max 0.382949, earliest_max 2024-01-24T01:50:00+00:00, mean 0.151446
 13. trace.to_ts-order-service.duration_p95 — count 72, max 0.141197, earliest_max 2024-01-24T02:06:00+00:00, mean 0.038606
 14. trace.to_ts-travel-service.duration_p95 — count 100, max 0.113557, earliest_max 2024-01-24T01:55:00+00:00, mean 0.04319
 15. trace.from_ts-order-service.duration_p95 — count 48, max 0.08098, earliest_max 2024-01-24T02:05:00+00:00, mean 0.014304
 16. trace.to_ts-travel-service.duration_mean — count 100, max 0.07709, earliest_max 2024-01-24T01:47:00+00:00, mean 0.033628
 17. trace.to_ts-order-service.duration_mean — count 72, max 0.072598, earliest_max 2024-01-24T01:57:00+00:00, mean 0.018147
 18. trace.from_ts-travel-service.duration_p95 — count 100, max 0.017395, earliest_max 2024-01-24T01:55:00+00:00, mean 0.009239
 19. trace.from_ts-travel-service.duration_mean — count 100, max 0.012963, earliest_max 2024-01-24T01:55:00+00:00, mean 0.00597
 20. trace.from_ts-order-service.duration_mean — count 48, max 0.011444, earliest_max 2024-01-24T02:05:00+00:00, mean 0.006917

- Error-rate traces: none of the selected trace_names that are error_rate have max_value > 0 (no entries returned).

- Stored variables for follow-up:
  - trace_candidates_df (filtered trace rows) — shape (2380, 4)
  - trace_agg_df (aggregates for each trace_name) — shape (40, 5)

If you want, I can: show the actual rows around the earliest max timestamps for any top trace_name, or correlate these trace spikes with the metric anomalies already found. Which would you like next?

The original code execution output of IPython Kernel is also provided below for reference:

(                                    trace_name  count   max_value  earliest_timestamp_of_max  mean_value
0          trace.to_ts-route-service.row_count     75  746.000000  2024-01-24T01:51:00+00:00  333.440000
1        trace.from_ts-route-service.row_count     25  746.000000  2024-01-24T01:51:00+00:00  666.880000
2        trace.from_ts-train-service.row_count     25  440.000000  2024-01-24T01:44:00+00:00  369.600000
3          trace.to_ts-train-service.row_count     75  440.000000  2024-01-24T01:44:00+00:00  154.000000
4         trace.from_ts-auth-service.row_count     25  392.000000  2024-01-24T01:51:00+00:00  341.640000
5       trace.from_ts-travel-service.row_count    100  392.000000  2024-01-24T01:53:00+00:00  135.180000
6         trace.to_ts-travel-service.row_count    100  392.000000  2024-01-24T01:53:00+00:00   91.990000
7           trace.to_ts-auth-service.row_count     50  392.000000  2024-01-24T01:51:00+00:00  213.580000
8        trace.from_ts-order-service.row_count     48   63.000000  2024-01-24T01:52:00+00:00   20.416667
9          trace.to_ts-order-service.row_count     72   63.000000  2024-01-24T01:52:00+00:00   15.027778
10       trace.to_ts-auth-service.duration_p95     50    1.112330  2024-01-24T01:47:00+00:00    0.331168
11      trace.to_ts-auth-service.duration_mean     50    0.382949  2024-01-24T01:50:00+00:00    0.151446
12      trace.to_ts-order-service.duration_p95     72    0.141197  2024-01-24T02:06:00+00:00    0.038606
13     trace.to_ts-travel-service.duration_p95    100    0.113557  2024-01-24T01:55:00+00:00    0.043190
14    trace.from_ts-order-service.duration_p95     48    0.080980  2024-01-24T02:05:00+00:00    0.014304
15    trace.to_ts-travel-service.duration_mean    100    0.077090  2024-01-24T01:47:00+00:00    0.033628
16     trace.to_ts-order-service.duration_mean     72    0.072598  2024-01-24T01:57:00+00:00    0.018147
17   trace.from_ts-travel-service.duration_p95    100    0.017395  2024-01-24T01:55:00+00:00    0.009239
18  trace.from_ts-travel-service.duration_mean    100    0.012963  2024-01-24T01:55:00+00:00    0.005970
19   trace.from_ts-order-service.duration_mean     48    0.011444  2024-01-24T02:05:00+00:00    0.006917, Empty DataFrame
Columns: [trace_name, max_value]
Index: [], (2380, 4), (40, 5))```
```

In [ ]:
# In[5]:

import pandas as pd
import numpy as np

# Reuse variables: log_df, candidates, start_window, end_window (already in kernel)

# 1) Filter logs to incident window and candidate services
log_candidates_df = log_df[
    (log_df['timestamp'] >= start_window) &
    (log_df['timestamp'] <= end_window) &
    (log_df['cmdb_id'].isin(candidates))
].copy()

# 2) Aggregate per (cmdb_id, log_name)
def agg_logs(g):
    cnt = len(g)
    maxv = g['value'].max()
    # earliest timestamp where value == max
    ts_max = g.loc[g['value'] == maxv, 'timestamp'].min()
    meanv = g['value'].mean()
    # count_positive and earliest_positive_timestamp
    pos_mask = g['value'] > 0
    cnt_pos = int(pos_mask.sum())
    ts_pos = None
    if cnt_pos > 0:
        ts_pos = g.loc[pos_mask, 'timestamp'].min()
    return pd.Series({
        'count': int(cnt),
        'max_value': float(maxv),
        'earliest_timestamp_of_max': ts_max,
        'mean_value': float(meanv),
        'count_positive': cnt_pos,
        'earliest_positive_timestamp': ts_pos
    })

log_agg_df = log_candidates_df.groupby(['cmdb_id', 'log_name'], dropna=False).apply(agg_logs).reset_index()

# Format timestamps as ISO UTC strings or None
def to_iso_or_none(ts):
    if pd.isna(ts) or ts is None:
        return None
    return ts.strftime('%Y-%m-%dT%H:%M:%S%z').replace('+0000', '+00:00')

log_agg_df['earliest_timestamp_of_max'] = log_agg_df['earliest_timestamp_of_max'].apply(to_iso_or_none)
log_agg_df['earliest_positive_timestamp'] = log_agg_df['earliest_positive_timestamp'].apply(to_iso_or_none)

# Round numeric columns for compactness
log_agg_df['max_value'] = log_agg_df['max_value'].round(6)
log_agg_df['mean_value'] = log_agg_df['mean_value'].round(6)

# Sort by count_positive desc then max_value desc and limit to top 20
log_agg_top20 = log_agg_df.sort_values(['count_positive', 'max_value'], ascending=[False, False]).head(20).reset_index(drop=True)

# 3) Candidate services that have any log.error_count > 0 during the window
error_mask = (log_candidates_df['log_name'] == 'log.error_count') & (log_candidates_df['value'] > 0)
error_services_df = (
    log_candidates_df[error_mask]
    .groupby('cmdb_id', dropna=False)['timestamp']
    .min()
    .reset_index()
    .rename(columns={'timestamp': 'earliest_positive_timestamp'})
)

# Format timestamp
if not error_services_df.empty:
    error_services_df['earliest_positive_timestamp'] = error_services_df['earliest_positive_timestamp'].apply(to_iso_or_none)

# Ensure columns ordering for aggregates
log_agg_df = log_agg_df[[
    'cmdb_id',
    'log_name',
    'count',
    'max_value',
    'earliest_timestamp_of_max',
    'mean_value',
    'count_positive',
    'earliest_positive_timestamp'
]]

# Output compact results and keep full DataFrames stored in variables
log_agg_top20, error_services_df, log_candidates_df.shape

```
Out[5]:
```
summary = (
    "Log summary (window 2024-01-24 01:40:52 — 2024-01-24 02:10:52 UTC):\n"
    "- Filtered logs for candidate services stored in log_candidates_df (198 rows).\n"
    "- Aggregates stored in log_agg_df. Top findings (sorted by count_positive then max_value):\n"
    "  * ts-travel-service / log.row_count: count=25, max=576.0, earliest_max=2024-01-24T02:02:00+00:00, mean=474.76, count_positive=25 (earliest positive 2024-01-24T01:43:00+00:00)\n"
    "  * ts-auth-service / log.row_count:  count=25, max=392.0, earliest_max=2024-01-24T01:51:00+00:00, mean=341.68, count_positive=25 (earliest positive 2024-01-24T01:43:00+00:00)\n"
    "  * ts-route-service / log.row_count: count=25, max=373.0, earliest_max=2024-01-24T01:51:00+00:00, mean=333.44, count_positive=25 (earliest positive 2024-01-24T01:43:00+00:00)\n"
    "  * ts-order-service / log.row_count: count=24, max=69.0,  earliest_max=2024-01-24T01:52:00+00:00, mean=36.58, count_positive=24 (earliest positive 2024-01-24T01:44:00+00:00)\n"
    "- For log.error_count across the candidate services, max_value = 0 for all (no positive error_count during the window). The list of services with log.error_count > 0 is empty.\n"
    "Overall takeaway: row_count signals are high and nonzero across the candidate services during the incident window, but there were no logged error_count events in that window. You can inspect log_candidates_df and log_agg_df for per-timestamp details."
)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(             cmdb_id         log_name  count  max_value  earliest_timestamp_of_max  mean_value  count_positive earliest_positive_timestamp
0  ts-travel-service    log.row_count     25      576.0  2024-01-24T02:02:00+00:00  474.760000              25   2024-01-24T01:43:00+00:00
1    ts-auth-service    log.row_count     25      392.0  2024-01-24T01:51:00+00:00  341.680000              25   2024-01-24T01:43:00+00:00
2   ts-route-service    log.row_count     25      373.0  2024-01-24T01:51:00+00:00  333.440000              25   2024-01-24T01:43:00+00:00
3   ts-order-service    log.row_count     24       69.0  2024-01-24T01:52:00+00:00   36.583333              24   2024-01-24T01:44:00+00:00
4    ts-auth-service  log.error_count     25        0.0  2024-01-24T01:43:00+00:00    0.000000               0                        None
5   ts-order-service  log.error_count     24        0.0  2024-01-24T01:44:00+00:00    0.000000               0                        None
6   ts-route-service  log.error_count     25        0.0  2024-01-24T01:43:00+00:00    0.000000               0                        None
7  ts-travel-service  log.error_count     25        0.0  2024-01-24T01:43:00+00:00    0.000000               0                        None, Empty DataFrame
Columns: [cmdb_id, earliest_positive_timestamp]
Index: [], (198, 4))```
```

In [ ]:
# In[6]:

import pandas as pd
import math

# Reuse existing DataFrames: metric_df, trace_df, log_df
# Define interval (UTC)
start = pd.to_datetime("2024-01-24T01:43:00", utc=True)
end = pd.to_datetime("2024-01-24T01:50:00", utc=True)

# 1) Metric rows for ts-train-service and specified KPIs
kpis = ['cpu','workload','socket','diskio','latency-50','latency-90']
metric_rows_df = metric_df[
    (metric_df['cmdb_id'] == 'ts-train-service') &
    (metric_df['kpi_name'].isin(kpis)) &
    (metric_df['timestamp'] >= start) &
    (metric_df['timestamp'] <= end)
].copy()

metric_rows_df['timestamp_iso'] = metric_rows_df['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%S%z').str.replace('+0000', '+00:00')
metric_rows_df = metric_rows_df[['timestamp_iso', 'kpi_name', 'value']].sort_values(['timestamp_iso', 'kpi_name']).reset_index(drop=True)

# 2) Trace rows for specified trace_names
trace_names = [
    'trace.from_ts-train-service.row_count',
    'trace.to_ts-train-service.row_count',
    'trace.to_ts-train-service.duration_mean',
    'trace.to_ts-train-service.duration_p95',
    'trace.to_ts-auth-service.duration_p95'
]
trace_rows_df = trace_df[
    (trace_df['trace_name'].isin(trace_names)) &
    (trace_df['timestamp'] >= start) &
    (trace_df['timestamp'] <= end)
].copy()

trace_rows_df['timestamp_iso'] = trace_rows_df['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%S%z').str.replace('+0000', '+00:00')
trace_rows_df = trace_rows_df[['timestamp_iso', 'trace_name', 'value']].sort_values(['timestamp_iso', 'trace_name']).reset_index(drop=True)

# 3) Log rows for ts-train-service and specified log_names
log_names = ['log.row_count', 'log.error_count']
log_rows_df = log_df[
    (log_df['cmdb_id'] == 'ts-train-service') &
    (log_df['log_name'].isin(log_names)) &
    (log_df['timestamp'] >= start) &
    (log_df['timestamp'] <= end)
].copy()

log_rows_df['timestamp_iso'] = log_rows_df['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%S%z').str.replace('+0000', '+00:00')
log_rows_df = log_rows_df[['timestamp_iso', 'log_name', 'value']].sort_values(['timestamp_iso']).reset_index(drop=True)

# Ensure total returned rows <= 50 by capping each table proportionally (keep relative sizes)
dfs = [metric_rows_df, trace_rows_df, log_rows_df]
lengths = [len(d) for d in dfs]
total = sum(lengths)
max_total = 50

if total <= max_total:
    # No trimming needed
    metric_rows_df_out, trace_rows_df_out, log_rows_df_out = metric_rows_df, trace_rows_df, log_rows_df
else:
    # Allocate rows proportionally, ensure at least 0 or 1 if original >0, and sum to max_total
    alloc = []
    for ln in lengths:
        if ln == 0:
            alloc.append(0)
        else:
            alloc.append(max(1, int(math.floor(ln / total * max_total))))
    # Adjust to reach exactly max_total
    current = sum(alloc)
    # If under-allocated, add to datasets with largest remaining until reach max_total
    remaining = max_total - current
    if remaining > 0:
        # compute remaining potentials
        potentials = [(i, lengths[i] - alloc[i]) for i in range(len(dfs))]
        # sort by potential desc (and by original size)
        potentials.sort(key=lambda x: (x[1], lengths[x[0]]), reverse=True)
        idx = 0
        while remaining > 0 and idx < len(potentials):
            i, pot = potentials[idx]
            if pot > 0:
                add = min(pot, remaining)
                alloc[i] += add
                remaining -= add
            idx += 1
        # If still remaining (all potentials exhausted), distribute 1s round-robin to non-empty until done
        idx = 0
        while remaining > 0:
            for i in range(len(alloc)):
                if lengths[i] > alloc[i]:
                    alloc[i] += 1
                    remaining -= 1
                    if remaining == 0:
                        break
    else:
        # If over-allocated, remove from smallest potential until sum == max_total
        over = -remaining
        potentials = [(i, alloc[i]) for i in range(len(dfs))]
        potentials.sort(key=lambda x: (x[1], -lengths[x[0]]))  # remove from smallest alloc first
        idx = 0
        while over > 0 and idx < len(potentials):
            i, a = potentials[idx]
            remove = min(a -  (1 if lengths[i]>0 else 0), over)
            if remove > 0:
                alloc[i] -= remove
                over -= remove
            idx += 1

    # Apply allocations
    metric_rows_df_out = metric_rows_df.head(alloc[0]).reset_index(drop=True)
    trace_rows_df_out = trace_rows_df.head(alloc[1]).reset_index(drop=True)
    log_rows_df_out = log_rows_df.head(alloc[2]).reset_index(drop=True)

# Final outputs (compact DataFrames)
metric_rows_df_out, trace_rows_df_out, log_rows_df_out

```
Out[6]:
```
Summary of raw rows in the critical interval 2024-01-24T01:43:00 — 2024-01-24T01:50:00 UTC (compact):

Returned rows
- Metric rows (ts-train-service, selected KPIs): 16 rows shown (part of the allotted 50-row cap).
- Trace rows (selected trace_names): 34 rows returned.
- Log rows (ts-train-service): none (no log.row_count or log.error_count rows in this interval).

Key findings — metrics (ts-train-service)
- CPU: low/normal in this interval (examples: 0.598087 at 01:43, 0.605579 at 01:44, 0.702056 at 01:45).
- Workload: elevated but modest (e.g., 2.54175 at 01:43 rising to 3.11125 at 01:44).
- Disk I/O: large absolute values and rising across timestamps (e.g., ~52,727 at 01:43 → ~56,264 at 01:45).
- Socket: steady around 9.
- Latency-50 / Latency-90: very small values in this interval (milliseconds-scale; e.g., latency-50 ~0.003–0.004 s).
- Overall: metrics show higher disk I/O and increasing workload but CPU remains moderate in this 01:43–01:50 window.

Key findings — traces
- Large row_count spikes targeting train/route:
  - trace.to_ts-train-service.row_count = 400 at 2024-01-24T01:45:00+00:00 (example).
  - trace.from_ts-train-service.row_count = 80 at 2024-01-24T01:43:00+00:00 (example).
  - Aggregates from earlier steps showed peaks up to ~440 around 01:44–01:45.
- Elevated tail latency observed to auth:
  - trace.to_ts-auth-service.duration_p95 = 0.307149 at 2024-01-24T01:43:00+00:00 (notably higher than other duration values in the window).
- trace.to_ts-train-service.duration_p95 values appear nonzero (small but measurable p95s, some entries ~0.001–0.0048 s).
- Overall: traces show traffic spikes into train/route services around 01:43–01:45 and a notable p95 spike toward auth at ~01:43.

Key findings — logs
- No log entries for ts-train-service in this interval (log_rows is empty). No error_count or row_count data returned for train in 01:43–01:50 from log.csv.

Compact interpretation / next steps
- The interval contains traffic spikes (high trace row_count) around 01:43–01:45 and an elevated auth p95 at ~01:43, while train metrics show increasing disk I/O and workload but only moderate CPU. No error logs exist in this slice.
- Recommend inspecting nearby timestamps (especially 01:47 where metric cpu anomalies were detected earlier) and correlating the trace row_count and auth p95 spikes with the metric spikes to confirm causal sequence.

If you want, I can (a) show the full untruncated metric_rows / trace_rows returned (or more rows), (b) pull raw metric rows specifically at 2024-01-24T01:47:00–01:48:00, or (c) correlate the exact timestamps of trace row_count peaks with metric anomalies. Which do you want next?

The original code execution output of IPython Kernel is also provided below for reference:

(                timestamp_iso    kpi_name         value
0   2024-01-24T01:43:00+00:00         cpu      0.598087
1   2024-01-24T01:43:00+00:00      diskio  52727.049796
2   2024-01-24T01:43:00+00:00  latency-50      0.004448
3   2024-01-24T01:43:00+00:00  latency-90      0.008917
4   2024-01-24T01:43:00+00:00      socket      9.000000
5   2024-01-24T01:43:00+00:00    workload      2.541750
6   2024-01-24T01:44:00+00:00         cpu      0.605579
7   2024-01-24T01:44:00+00:00      diskio  54143.919850
8   2024-01-24T01:44:00+00:00  latency-50      0.003945
9   2024-01-24T01:44:00+00:00  latency-90      0.008477
10  2024-01-24T01:44:00+00:00      socket      9.000000
11  2024-01-24T01:44:00+00:00    workload      3.111250
12  2024-01-24T01:45:00+00:00         cpu      0.702056
13  2024-01-24T01:45:00+00:00      diskio  56264.250220
14  2024-01-24T01:45:00+00:00  latency-50      0.003882
15  2024-01-24T01:45:00+00:00  latency-90      0.008439,                 timestamp_iso                               trace_name       value
0   2024-01-24T01:43:00+00:00    trace.from_ts-train-service.row_count   80.000000
1   2024-01-24T01:43:00+00:00    trace.to_ts-auth-service.duration_p95    0.307149
2   2024-01-24T01:43:00+00:00    trace.to_ts-auth-service.duration_p95    0.001629
3   2024-01-24T01:43:00+00:00  trace.to_ts-train-service.duration_mean    0.000810
4   2024-01-24T01:43:00+00:00  trace.to_ts-train-service.duration_mean    0.003603
..                        ...                                      ...         ...
29  2024-01-24T01:45:00+00:00  trace.to_ts-train-service.duration_mean    0.003464
30  2024-01-24T01:45:00+00:00   trace.to_ts-train-service.duration_p95    0.001183
31  2024-01-24T01:45:00+00:00   trace.to_ts-train-service.duration_p95    0.004818
32  2024-01-24T01:45:00+00:00   trace.to_ts-train-service.duration_p95    0.004080
33  2024-01-24T01:45:00+00:00      trace.to_ts-train-service.row_count  400.000000

[34 rows x 3 columns], Empty DataFrame
Columns: [timestamp_iso, log_name, value]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[7]:

import math
import pandas as pd

# Reuse existing DataFrames and variables in kernel: metric_df, trace_df, log_df, candidates

# Define focused interval (UTC)
start = pd.to_datetime("2024-01-24T01:46:00", utc=True)
end = pd.to_datetime("2024-01-24T01:48:00", utc=True)

# 1) Metric rows for ts-train-service and specified KPIs
kpis = ['cpu','workload','socket','diskio','latency-50','latency-90']
metric_rows = metric_df[
    (metric_df['cmdb_id'] == 'ts-train-service') &
    (metric_df['kpi_name'].isin(kpis)) &
    (metric_df['timestamp'] >= start) &
    (metric_df['timestamp'] <= end)
].copy()

metric_rows['timestamp'] = metric_rows['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%S%z').str.replace('+0000', '+00:00')
metric_rows = metric_rows[['timestamp', 'kpi_name', 'value']].sort_values(['timestamp', 'kpi_name']).reset_index(drop=True)

# 2) Trace rows for specified trace_names
trace_names = [
    'trace.from_ts-train-service.row_count',
    'trace.to_ts-train-service.row_count',
    'trace.to_ts-train-service.duration_mean',
    'trace.to_ts-train-service.duration_p95',
    'trace.to_ts-auth-service.duration_p95'
]
trace_rows = trace_df[
    (trace_df['trace_name'].isin(trace_names)) &
    (trace_df['timestamp'] >= start) &
    (trace_df['timestamp'] <= end)
].copy()

trace_rows['timestamp'] = trace_rows['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%S%z').str.replace('+0000', '+00:00')
trace_rows = trace_rows[['timestamp', 'trace_name', 'value']].sort_values(['timestamp', 'trace_name']).reset_index(drop=True)

# 3) Log rows for candidate services and specified log_names
log_names = ['log.row_count', 'log.error_count']
log_rows = log_df[
    (log_df['cmdb_id'].isin(candidates)) &
    (log_df['log_name'].isin(log_names)) &
    (log_df['timestamp'] >= start) &
    (log_df['timestamp'] <= end)
].copy()

log_rows['timestamp'] = log_rows['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%S%z').str.replace('+0000', '+00:00')
log_rows = log_rows[['timestamp', 'cmdb_id', 'log_name', 'value']].sort_values(['timestamp', 'cmdb_id']).reset_index(drop=True)

# Ensure total returned rows <= 50 by capping each table proportionally
dfs = [metric_rows, trace_rows, log_rows]
lengths = [len(d) for d in dfs]
total = sum(lengths)
max_total = 50

if total <= max_total:
    metric_rows_out, trace_rows_out, log_rows_out = metric_rows, trace_rows, log_rows
else:
    # initial proportional allocation (at least 1 for non-empty)
    alloc = []
    for ln in lengths:
        if ln == 0:
            alloc.append(0)
        else:
            alloc.append(max(1, int(math.floor(ln / total * max_total))))
    # adjust to exactly max_total
    current = sum(alloc)
    remaining = max_total - current
    if remaining > 0:
        # distribute to datasets with largest remaining capacity
        potentials = [(i, lengths[i] - alloc[i]) for i in range(len(dfs))]
        potentials.sort(key=lambda x: (x[1], lengths[x[0]]), reverse=True)
        idx = 0
        while remaining > 0 and idx < len(potentials):
            i, pot = potentials[idx]
            if pot > 0:
                add = min(pot, remaining)
                alloc[i] += add
                remaining -= add
            idx += 1
        # if still remaining, give extra rows round-robin to those that can accept more
        idx = 0
        while remaining > 0:
            for i in range(len(alloc)):
                if lengths[i] > alloc[i]:
                    alloc[i] += 1
                    remaining -= 1
                    if remaining == 0:
                        break
    elif remaining < 0:
        # remove excess from smallest allocations if possible
        over = -remaining
        potentials = [(i, alloc[i] - (1 if lengths[i]>0 else 0)) for i in range(len(dfs))]
        potentials.sort(key=lambda x: x[1])  # remove from smallest capacity first
        idx = 0
        while over > 0 and idx < len(potentials):
            i, removable = potentials[idx]
            if removable > 0:
                remove = min(removable, over)
                alloc[i] -= remove
                over -= remove
            idx += 1

    metric_rows_out = metric_rows.head(alloc[0]).reset_index(drop=True)
    trace_rows_out = trace_rows.head(alloc[1]).reset_index(drop=True)
    log_rows_out = log_rows.head(alloc[2]).reset_index(drop=True)

# Final returned DataFrames (compact)
metric_rows_out, trace_rows_out, log_rows_out

```
Out[7]:
```
Summary for interval 2024-01-24T01:46:00 — 2024-01-24T01:48:00 UTC:

Metric observations (ts-train-service)
- CPU: a clear spike at 01:47 — CPU = 1.606954 (vs ~0.63 at 01:46).
- Disk I/O: high absolute values and rising (≈49,397 at 01:46 → ≈52,246 at 01:47).
- Workload: present and elevated at 01:46 (~2.91); not listed at 01:47 in the returned sample.
- Socket: steady around 9–9.33.
- Latencies (latency-50/90): very small absolute values (~0.0039–0.0095 s).
Takeaway: a CPU spike at 01:47 co-occurs with high and increasing disk I/O and sustained workload.

Trace observations
- Large traffic spikes into train/route around 01:46–01:47:
  - trace.from_ts-train-service.row_count = 320 at 01:46.
  - trace.to_ts-train-service.row_count entries at 01:47 include 280 and other bursts (multiple row_count values shown).
- Notable auth tail-latency spike at 01:46:
  - trace.to_ts-auth-service.duration_p95 = 0.492350 at 01:46 (substantially larger than other duration values in the slice).
Takeaway: heavy request volume into train/route services around 01:46–01:47, and an elevated p95 observed for calls to auth at ~01:46.

Log observations (candidate services)
- High row_count values present for multiple services during this interval (examples at 01:46):
  - ts-travel-service log.row_count = 482
  - ts-auth-service log.row_count = 368
  - ts-route-service log.row_count = 336
  - ts-order-service log.row_count = 41
- log.error_count values are 0 for these timestamps (no logged errors in this slice).
Note: no ts-train-service log rows were returned for this interval.

Overall compact interpretation
- A traffic surge (large trace row_count) into train/route around 01:46–01:47 aligns with a CPU spike on ts-train-service at 01:47 and rising disk I/O. There is also a pronounced auth p95 spike at ~01:46. Logs show increased request volume across services but no error_count events in this window.
- Suggested next step: correlate the 01:46 auth p95 spike and the trace row_count peaks with the ts-train-service CPU/disk spikes (inspect raw rows at 01:46–01:48 and adjacent minutes) to confirm causal ordering.

The original code execution output of IPython Kernel is also provided below for reference:

(                    timestamp    kpi_name         value
0   2024-01-24T01:46:00+00:00         cpu      0.629882
1   2024-01-24T01:46:00+00:00      diskio  49396.802527
2   2024-01-24T01:46:00+00:00  latency-50      0.003953
3   2024-01-24T01:46:00+00:00  latency-90      0.008499
4   2024-01-24T01:46:00+00:00      socket      9.333333
5   2024-01-24T01:46:00+00:00    workload      2.906717
6   2024-01-24T01:47:00+00:00         cpu      1.606954
7   2024-01-24T01:47:00+00:00      diskio  52245.996021
8   2024-01-24T01:47:00+00:00  latency-50      0.004438
9   2024-01-24T01:47:00+00:00  latency-90      0.009502
10  2024-01-24T01:47:00+00:00      socket      9.000000,                     timestamp                               trace_name       value
0   2024-01-24T01:46:00+00:00    trace.from_ts-train-service.row_count  320.000000
1   2024-01-24T01:46:00+00:00    trace.to_ts-auth-service.duration_p95    0.492350
2   2024-01-24T01:46:00+00:00    trace.to_ts-auth-service.duration_p95    0.001905
3   2024-01-24T01:46:00+00:00  trace.to_ts-train-service.duration_mean    0.000874
4   2024-01-24T01:46:00+00:00  trace.to_ts-train-service.duration_mean    0.004871
..                        ...                                      ...         ...
19  2024-01-24T01:47:00+00:00   trace.to_ts-train-service.duration_p95    0.006581
20  2024-01-24T01:47:00+00:00   trace.to_ts-train-service.duration_p95    0.005235
21  2024-01-24T01:47:00+00:00      trace.to_ts-train-service.row_count  280.000000
22  2024-01-24T01:47:00+00:00      trace.to_ts-train-service.row_count   35.000000
23  2024-01-24T01:47:00+00:00      trace.to_ts-train-service.row_count   35.000000

[24 rows x 3 columns],                     timestamp            cmdb_id         log_name  value
0   2024-01-24T01:46:00+00:00    ts-auth-service  log.error_count      0
1   2024-01-24T01:46:00+00:00    ts-auth-service    log.row_count    368
2   2024-01-24T01:46:00+00:00   ts-order-service  log.error_count      0
3   2024-01-24T01:46:00+00:00   ts-order-service    log.row_count     41
4   2024-01-24T01:46:00+00:00   ts-route-service  log.error_count      0
5   2024-01-24T01:46:00+00:00   ts-route-service    log.row_count    336
6   2024-01-24T01:46:00+00:00  ts-travel-service  log.error_count      0
7   2024-01-24T01:46:00+00:00  ts-travel-service    log.row_count    482
8   2024-01-24T01:47:00+00:00    ts-auth-service  log.error_count      0
9   2024-01-24T01:47:00+00:00    ts-auth-service    log.row_count    316
10  2024-01-24T01:47:00+00:00   ts-order-service  log.error_count      0
11  2024-01-24T01:47:00+00:00   ts-order-service    log.row_count     40
12  2024-01-24T01:47:00+00:00   ts-route-service  log.error_count      0
13  2024-01-24T01:47:00+00:00   ts-route-service    log.row_count    312
14  2024-01-24T01:47:00+00:00  ts-travel-service  log.error_count      0)

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```